# Experiment 2 analysis notebook

Interactive exploration of the rolling inverse-PINN vs least-squares
comparison. Reads only saved artifacts from `results/experiment_2/` —
no recomputation, so numbers always match the recorded experiment.

Run from the repository root (`jupyter notebook notebooks/`).

In [ ]:
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

table = pd.read_csv(ROOT / 'results/experiment_2/all_block_tables.csv')
sigma = pd.read_csv(ROOT / 'results/experiment_2/all_sigma_histories.csv')
summary = json.loads((ROOT / 'results/experiment_2/summary.json').read_text())
print('conditions:', summary['conditions'])
print('completed runs:', summary.get('n_completed_runs'), '/', summary.get('n_expected_runs'))
table.head()

## Pooled RMSE by condition and method

Pooled across blocks and seeds. Seed spread of the mean block RMSE is
shown in the `seed_std` column of the long table below.

In [ ]:
conds = ['constant_clean','constant_noise','constant_sparse','constant_noise_sparse','gradual_change','abrupt_change']
conds = [c for c in conds if c in set(table['condition'])]
methods = ['least_squares','pinn','frozen','data_only_nn']
rows = []
for cond in conds:
    for meth in methods:
        sub = table[(table.condition==cond)&(table.method==meth)]
        if not len(sub): continue
        seed_rmse = sub.groupby('seed')['rmse'].mean()
        rows.append(dict(condition=cond, method=meth,
                         pooled_rmse=float(np.sqrt(np.mean(sub.rmse**2))),
                         mean_block_rmse=float(sub.rmse.mean()),
                         seed_std=float(seed_rmse.std(ddof=1))))
long = pd.DataFrame(rows)
pivot = long.pivot(index='condition', columns='method', values='pooled_rmse')
display_order = [c for c in conds if c in pivot.index]
pivot.loc[display_order].round(4)

## Sigma estimates vs truth (abrupt change, seed 0)

The block-effective reference is the integrated volatility over the
upcoming evaluation block — the quantity a perfect calibrator would use.

In [ ]:
sub = sigma[(sigma.condition=='abrupt_change')&(sigma.seed==0)].sort_values('cutoff')
fig, ax = plt.subplots(figsize=(9,4))
ax.plot(sub.cutoff, sub.sigma_ref_instantaneous, 'k-', lw=2, label='true instantaneous')
ax.plot(sub.cutoff, sub.sigma_ref_effective_block, 'k:', lw=2, label='true block-effective')
ax.plot(sub.cutoff, sub.sigma_ls, label='LS')
ax.plot(sub.cutoff, sub.sigma_pinn, label='PINN')
ax.plot(sub.cutoff, sub.sigma_frozen, label='frozen')
ax.legend(); ax.set_ylabel('sigma'); ax.grid(alpha=.3)
plt.xticks(rotation=30); plt.tight_layout(); plt.show()

## Paired block differences (PINN − LS)

Positive difference means PINN worse on that block. Blocks are
chronological and possibly autocorrelated; treat any intervals as
descriptive only.

In [ ]:
fig, ax = plt.subplots(figsize=(9,4))
for cond in conds:
    s = table[(table.condition==cond)&(table.seed==0)]
    p = s[s.method=='pinn'].sort_values('event_id')
    l = s[s.method=='least_squares'].sort_values('event_id')
    n = min(len(p), len(l))
    if n:
        ax.plot(p.event_id.to_numpy()[:n], p.rmse.to_numpy()[:n]-l.rmse.to_numpy()[:n], 'o-', ms=3, label=cond)
ax.axhline(0, color='k', lw=1); ax.legend(fontsize=8)
ax.set_xlabel('walk-forward event'); ax.set_ylabel('RMSE(PINN) - RMSE(LS)')
ax.grid(alpha=.3); plt.tight_layout(); plt.show()

## Gate 5: constant-volatility recovery


In [ ]:
print(json.dumps(summary['gate5_constant_recovery'], indent=2))